<a href="https://colab.research.google.com/github/AsserGharib1/flyrank-internshipML/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsserGharib1/flyrank-internshipML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup and access

The warehouse is gated so this needs a Hugging Face read token.

I keep the token in the Colab Secrets panel under the name `HF_TOKEN`. It never goes in a cell,
because my repo is public and anything in a cell ends up on GitHub.

I also stay away from the table ending in `_sample`. It only holds June 2026, which is the last
month there is. My label looks at what happens next, so if I built it there my outcome window would
be the same month I was supposed to be testing on later. I use March and April instead and leave
June alone.

In [1]:
%pip -q install duckdb pandas scikit-learn

import os, duckdb, pandas as pd, numpy as np

# Token comes from the Colab secrets panel. Never typed in here.
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, (
    "No HF_TOKEN. Open the key icon on the left, add a secret called HF_TOKEN, turn on notebook "
    "access. Use a plain Read token, not a fine grained one."
)
print("Token found. Printing the length only:", len(HF_TOKEN))

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT = BASE + "/fact_content_daily_performance"
FEATURE_MONTH = "2026-03"
OUTCOME_MONTH = "2026-04"

def month_rel(month):
    return f"read_parquet('{FACT}/month={month}/*.parquet')"

print("Feature month:", FEATURE_MONTH, "  Outcome month:", OUTCOME_MONTH)

Token found. Printing the length only: 37
Feature month: 2026-03   Outcome month: 2026-04


Before writing any query I wanted to see the real column names instead of guessing them. This is
not one of my three verification queries, it just reads the schema.

Good thing I did. I had expected something like `impressions` and `avg_position`. The actual names
are `gsc_impressions` and `gsc_avg_position`, and there is also a `gsc_sum_position` column that
turned out to matter later.

In [2]:
# Just looking at what columns actually exist before I write anything real.

schema = con.sql(f"DESCRIBE SELECT * FROM {month_rel(FEATURE_MONTH)} LIMIT 1").df()
print(schema[["column_name", "column_type"]].to_string(index=False))

available = set(schema.column_name)
for needed in ["report_date", "client_hash_id", "content_hash_id",
               "gsc_impressions", "gsc_clicks", "gsc_sum_position", "ga4_data_available"]:
    assert needed in available, f"Column {needed} is missing. Columns are: {sorted(available)}"
print()
print("All the columns I need are there.")

             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_copilot      BIGINT
 

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row is one piece of content on one client site, looked at on 1 April 2026.

I picked 1 April because that is the moment a reviewer would sit down and decide what to open. So
everything I am allowed to know has to have happened before that date.

That gives me two windows.

March 2026 is where my features come from. April 2026 is what I am trying to predict. They do not
touch, which is the whole point.

The table itself is at a different grain from my row. It is one row per day per client per content.
Mine is one row per content item. So I add March up to get one row per item, add April up separately
to get the outcome, and join them. Query 1 checks the table really is at the grain I think it is
before I aggregate anything on top of it.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

The contract in five answers first.

**One row** is one content item on one client site, as it stood on 1 April 2026.

**Tables.** `fact_content_daily_performance` for both windows, and `dim_clients` once at the end for
the limitation check.

**Time window.** Features from March 2026, outcome from April 2026.

**What I predict.** Whether a page drops at least 20 points more than the middle page on the same
site, comparing April impressions against March. Same target I settled on in ML-03. I did not just
assume the minus 20 still worked here though, I printed five different cut offs and looked.

**What I exclude on purpose.** All the Analytics columns.

Now the buckets.

**Features.** March impressions, March clicks, March active days, March average position, and
movement inside March. Five, which is the cap.

**Label.** April impressions and anything built from them. The change against March, the gap against
the site median, and the final yes or no.

**Context.** The client hash, the content hash and the date. For joining, grouping and holding whole
clients out of the test. The model never sees them.

**Excluded.** The Analytics columns. I nearly used sessions and engagement, because on the surface
they look like richer signals than plain impressions. Then I ran query 3. Only 4.2 percent of March
rows actually have Analytics switched on. The other 95 percent are zeros with a flag saying the
tracking was not there. If I fed those in, the model would learn which clients had Analytics
configured, which tells me about the client's setup and nothing about the page.

I am also leaving out `fact_content_query_90d`. Its window is a fixed 90 days that runs straight
into my April outcome window, so it would need proper window alignment first and that is not this
notebook's job.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three queries. Grain, then size and dates, then availability. After that the five features and the
leak experiment.

In [3]:
# QUERY 1 of 3. Is the table really one row per day per client per content?
# If it is, grouping by those three and asking for groups bigger than one gives nothing back.

grain = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_in_group
    FROM {month_rel(FEATURE_MONTH)}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("QUERY 1  grain probe on", FEATURE_MONTH)
print(f"  duplicate groups: {len(grain)}")
print("  grain holds, safe to aggregate" if len(grain) == 0 else "  grain is broken, stop here")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

QUERY 1  grain probe on 2026-03
  duplicate groups: 0
  grain holds, safe to aggregate


Zero duplicates, so the grain is what I said and I can safely roll it up.

Query 2 next. I want the row count and the date range, mostly to check the month partition actually
contains only that month.

In [4]:
# QUERY 2 of 3. How big is my slice and does it stay inside March?

span = con.sql(f"""
    SELECT COUNT(*)                          AS rows_in_month,
           COUNT(DISTINCT client_hash_id)    AS clients,
           COUNT(DISTINCT content_hash_id)   AS content_items,
           MIN(report_date)                  AS first_date,
           MAX(report_date)                  AS last_date
    FROM {month_rel(FEATURE_MONTH)}
""").df()

print("QUERY 2  size and dates for", FEATURE_MONTH)
print(span.to_string(index=False))

row = span.iloc[0]
# First time round I compared these as strings and got False, which confused me for a while.
# The dates come back as timestamps, so str() gives "2026-03-31 00:00:00", and that sorts
# after "2026-03-31". Comparing real dates instead of text fixes it.
first = pd.Timestamp(row.first_date).date()
last  = pd.Timestamp(row.last_date).date()
inside = first >= pd.Timestamp("2026-03-01").date() and last <= pd.Timestamp("2026-03-31").date()
print()
print(f"  first {first}, last {last}, stays inside March: {inside}")

QUERY 2  size and dates for 2026-03
 rows_in_month  clients  content_items first_date  last_date
       9841378       55         331437 2026-03-01 2026-03-31

  first 2026-03-01, last 2026-03-31, stays inside March: True


9.8 million rows for one month, 55 clients, 331,437 content items, and the dates run 1 March to 31
March. So the partitioning is doing what the docs say.

That row count is also a reminder of why I am aggregating in SQL and only pulling the small result
back. Loading 9.8 million rows into pandas for one month would be silly, and there are 17 months.

Query 3 is the one that decides whether I use the Analytics columns.

In [5]:
# QUERY 3 of 3. How much of the month actually has Analytics data, filtered with IS TRUE?

avail = con.sql(f"""
    SELECT COUNT(*)                                        AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_with_analytics,
           COUNT(*) FILTER (WHERE gsc_impressions > 0)      AS rows_with_impressions
    FROM {month_rel(FEATURE_MONTH)}
""").df()

print("QUERY 3  availability on", FEATURE_MONTH)
print(avail.to_string(index=False))

a = avail.iloc[0]
print()
print(f"  rows left after an Analytics IS TRUE filter : {a.rows_with_analytics:,} "
      f"({a.rows_with_analytics/a.total_rows:.1%})")
print(f"  rows with any search impressions            : {a.rows_with_impressions:,} "
      f"({a.rows_with_impressions/a.total_rows:.1%})")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

QUERY 3  availability on 2026-03
 total_rows  rows_with_analytics  rows_with_impressions
    9841378               413966                3611061

  rows left after an Analytics IS TRUE filter : 413,966 (4.2%)
  rows with any search impressions            : 3,611,061 (36.7%)


That settled it. Analytics survives on 4.2 percent of rows and search on 36.7 percent. So Analytics
is out, and now I have a number behind that decision instead of just a feeling.

### The five features

All five come from March only, so all of them had already happened by 1 April. That is the test I am
applying to each one.

* `impressions_prev30`, total search impressions in March. Knowable on 1 April because March was
  over.
* `clicks_prev30`, total search clicks in March. Same reason.
* `active_days_prev30`, how many days in March the page picked up any impressions. Only counts days
  inside the feature window.
* `avg_position_prev30`, the average search position across March. See the note below, I got this
  one wrong at first.
* `momentum_in_march`, second half of March against the first half. Both halves sit inside the
  feature window, so it gives me a sense of direction without touching April.

On position. My first version did `AVG(gsc_avg_position)` across the daily rows. Then I remembered
the schema also had `gsc_sum_position`, and realised averaging a daily average treats a day with 2
impressions the same as a day with 2,000. So I switched to total position divided by total
impressions, which weights each day by how much it actually mattered. Same idea, better number.

I also print the blanks under the frame, because the section title asks about missing values and
because I noticed my model was running on fewer rows than the frame had.

In [6]:
# Build the frame. March gives me the features, April gives me the outcome.

FLOOR = 100   # a page needs at least this many March impressions or April movement is just noise

frame = con.sql(f"""
    WITH march AS (
        SELECT client_hash_id  AS client_id,
               content_hash_id AS content_id,
               SUM(gsc_impressions)                    AS impressions_prev30,
               SUM(gsc_clicks)                         AS clicks_prev30,
               COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_days_prev30,
               -- weighted by impressions, not an average of averages
               SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_position_prev30,
               SUM(gsc_impressions) FILTER (WHERE EXTRACT(day FROM report_date) > 15)  AS late_march,
               SUM(gsc_impressions) FILTER (WHERE EXTRACT(day FROM report_date) <= 15) AS early_march
        FROM {month_rel(FEATURE_MONTH)}
        GROUP BY 1, 2
    ),
    april AS (
        SELECT client_hash_id  AS client_id,
               content_hash_id AS content_id,
               SUM(gsc_impressions) AS impressions_next30
        FROM {month_rel(OUTCOME_MONTH)}
        GROUP BY 1, 2
    )
    SELECT m.client_id,
           m.content_id,
           m.impressions_prev30,
           m.clicks_prev30,
           m.active_days_prev30,
           m.avg_position_prev30,
           CASE WHEN m.early_march > 0
                THEN (m.late_march - m.early_march) * 100.0 / m.early_march
                ELSE NULL END              AS momentum_in_march,
           COALESCE(a.impressions_next30, 0) AS impressions_next30
    FROM march m
    LEFT JOIN april a USING (client_id, content_id)
    WHERE m.impressions_prev30 >= {FLOOR}
    -- sorted so a rerun reads the rows in the same order and gives the same score
    ORDER BY client_id, content_id
""").df()

print(f"{len(frame):,} pages from {frame.client_id.nunique()} clients cleared the {FLOOR} impression floor")
print(f"rows per content item: {len(frame)/frame.content_id.nunique():.2f}  (1.00 means the grain survived the join)")
print()
print(frame.head(5).to_string(index=False))
print()

# Missing values, which the section title asks about.
print("Which columns go blank, and is it random?")
for col in ["impressions_prev30", "clicks_prev30", "active_days_prev30",
            "avg_position_prev30", "momentum_in_march"]:
    print(f"  {col:<22} blank on {frame[col].isna().mean():6.2%} of rows")

blank = frame[frame.momentum_in_march.isna()]
print()
print(f"Only momentum goes blank, on {len(blank):,} pages.")
print("Checking whether those pages look different from the rest:")
print(f"  median March impressions, blank pages : {blank.impressions_prev30.median():,.0f}")
print(f"  median March impressions, the rest    : {frame[frame.momentum_in_march.notna()].impressions_prev30.median():,.0f}")
print(f"  median active days, blank pages       : {blank.active_days_prev30.median():.0f}")
print(f"  median active days, the rest          : {frame[frame.momentum_in_march.notna()].active_days_prev30.median():.0f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101,441 pages from 44 clients cleared the 100 impression floor
rows per content item: 1.00  (1.00 means the grain survived the join)

              client_id               content_id  impressions_prev30  clicks_prev30  active_days_prev30  avg_position_prev30  momentum_in_march  impressions_next30
client_0797ff3a1fc9a6a5 content_04c67f3541177192               331.0            2.0                  31            14.377644          78.151261               561.0
client_0797ff3a1fc9a6a5 content_0f30e04e709c7b5d               145.0            0.0                  30             8.124138           1.388889                23.0
client_0797ff3a1fc9a6a5 content_1207efddce873942               461.0            0.0                  31            14.488069         -37.102473               964.0
client_0797ff3a1fc9a6a5 content_167472cd0802a8f3               232.0            0.0                  29            11.961207         151.515152               248.0
client_0797ff3a1fc9a6a5 content_27f8100281413b

In [7]:
# Now the label. I kept the minus 20 from ML-03 but I wanted to see the alternatives first.

frame["change_pct"] = (
    (frame.impressions_next30 - frame.impressions_prev30) / frame.impressions_prev30 * 100
)
frame["gap_vs_client"] = frame.change_pct - frame.groupby("client_id").change_pct.transform("median")

print(f"middle change across all pages : {frame.change_pct.median():.1f}%")
print(f"client medians run from {frame.groupby('client_id').change_pct.median().min():.1f}% "
      f"to {frame.groupby('client_id').change_pct.median().max():.1f}%")
print()
for cut in (-10, -15, -20, -25, -30):
    print(f"  gap of {cut:>4} points or worse -> positive rate {(frame.gap_vs_client <= cut).mean():.3f}")

GAP_CUTOFF = -20
frame["target_falls_behind"] = (frame.gap_vs_client <= GAP_CUTOFF).astype(int)
print()
print(f"Sticking with {GAP_CUTOFF}. Base rate {frame.target_falls_behind.mean():.3f}")

middle change across all pages : -22.1%
client medians run from -100.0% to 696.0%

  gap of  -10 points or worse -> positive rate 0.398
  gap of  -15 points or worse -> positive rate 0.344
  gap of  -20 points or worse -> positive rate 0.290
  gap of  -25 points or worse -> positive rate 0.242
  gap of  -30 points or worse -> positive rate 0.197

Sticking with -20. Base rate 0.290


Four of my five features are never blank. Only `momentum_in_march` goes missing, and it is not
random. It is blank exactly when a page had no impressions in the first half of March, so there is
nothing to compare the second half against. Those pages are much smaller and much less active than
the rest, which the numbers above show.

That matters because dropping them is not a neutral act. It quietly removes the quietest pages from
the model, and those are arguably the ones an editor would most want flagged. For now I drop them so
the score is honest about what it was trained on. In ML-06 the better answer is probably to fill the
blank with zero and add a flag saying the page was dormant early in the month, so the model can use
that fact instead of losing the row.

The middle page lost 22 percent of its impressions between March and April, which lines up with
what I saw in the starter file. Minus 20 gives a base rate of 0.29, which sits in a sensible place
between the five options, so I kept it.

One thing I did not expect. The client medians run from minus 100 percent all the way up to plus 696
percent. A site whose middle page grew 696 percent almost certainly has very few pages in my slice,
so its median is jumping around on tiny numbers. I am not fixing that here, but it is a reason to
look at a minimum pages per client rule in ML-06.

### The trap

The card asks me to break it on purpose. So I am adding April impressions as a feature. My label is
calculated from that column, so the model is being handed the answer and the score should shoot up.

I already did this by accident in ML-03. My first feature list there was five columns measured over
90 days while the outcome sat inside those same 90 days. Doing it deliberately here is the point.
The thing to remember is that a score that suddenly looks great is a reason to go and check your
windows.

In [8]:
# Score it twice. Once cheating, once not. Whole clients held out of the test both times.

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

HONEST_FEATURES = ["impressions_prev30", "clicks_prev30", "active_days_prev30",
                   "avg_position_prev30", "momentum_in_march"]
LEAKY_FEATURE = "impressions_next30"   # the label is calculated from this column

model_df = frame.dropna(subset=HONEST_FEATURES).copy()
y = model_df.target_falls_behind
groups = model_df.client_id

train_idx, test_idx = next(
    GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0).split(model_df, y, groups)
)

def quick_score(columns):
    X = model_df[columns]
    # n_jobs=1 on purpose. With all cores the score wobbled in the third decimal between
    # runs, which made it impossible to quote a number in my write up and have it stay true.
    m = RandomForestClassifier(n_estimators=120, min_samples_leaf=20, random_state=0, n_jobs=1)
    m.fit(X.iloc[train_idx], y.iloc[train_idx])
    return roc_auc_score(y.iloc[test_idx], m.predict_proba(X.iloc[test_idx])[:, 1])

print(f"{len(model_df):,} rows, {groups.iloc[test_idx].nunique()} clients held out for testing")
print(f"base rate {y.mean():.3f}")
print()

leaked = quick_score(HONEST_FEATURES + [LEAKY_FEATURE])
print(f"with April impressions in there : {leaked:.3f}")

honest = quick_score(HONEST_FEATURES)
print(f"without it                      : {honest:.3f}")
print()
print(f"So the leak was worth {leaked - honest:+.3f}, and none of it was real.")

model_df = model_df.drop(columns=[LEAKY_FEATURE])
print(f"Dropped it. Still in the frame? {LEAKY_FEATURE in model_df.columns}")
print()
print("Keeping these five for the modelling weeks:")
for f in HONEST_FEATURES:
    print("   ", f)

96,671 rows, 12 clients held out for testing
base rate 0.298

with April impressions in there : 0.924
without it                      : 0.669

So the leak was worth +0.254, and none of it was real.
Dropped it. Still in the frame? False

Keeping these five for the modelling weeks:
    impressions_prev30
    clicks_prev30
    active_days_prev30
    avg_position_prev30
    momentum_in_march


About 0.92 with the leak in and about 0.67 without it. So roughly a quarter of that score was
coming from a column that simply held the answer.

The honest number is the one I carry forward. Against a base rate of 0.30 it is doing something, but
it is nowhere near finished, and it still has to beat a plain rule baseline in ML-07 before I would
say a model is worth the trouble.

Two things I learned running this. Fixing the position feature nudged the honest score up by about a
hundredth, which is small but went the right way. And the first time I ran it with all cores the
score moved slightly between runs, so I pinned it to one core. A number I quote in writing should
still be true the next time someone presses run.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The limit I want to name is that March means different things for different clients.

Everyone did not start being tracked on the same day. So when I compare a page at one site against a
page at another, I am using the same calendar month but a very different amount of history behind
it. A client three months into tracking is still ramping up coverage, so their March can look
unusually full or unusually thin, and then April looks like a drop that is really just measurement
settling down.

The query below counts how bad this is in my slice.

I am not correcting for it here. In ML-06 my options are to drop the clients with short history, or
to give each client its own window instead of one shared calendar month.

Three other things this data cannot tell me, worth writing down so I do not forget later.

* Why a page fell. I only see impressions, clicks and position, never the reason.
* Whether editing a page would help. Nothing here records what happens after someone makes a change,
  so I cannot say anything about cause without a real experiment.
* Anything about an actual client, address or search term. All of that was stripped before release.

In [9]:
# How uneven is the history across clients?

hist = con.sql(f"""
    SELECT COUNT(*)                                                  AS clients,
           COUNT(*) FILTER (WHERE gsc_data_start > DATE '2025-09-01') AS started_recently,
           MIN(gsc_data_start) AS earliest_start,
           MAX(gsc_data_start) AS latest_start
    FROM read_parquet('{BASE}/dim_clients.parquet')
    WHERE gsc_data_start IS NOT NULL
""").df()

print(hist.to_string(index=False))
h = hist.iloc[0]
print()
print(f"{h.started_recently} of {h.clients} clients only started search tracking after September 2025.")
print("Same calendar month for everyone, very different amounts of history behind it.")

 clients  started_recently earliest_start latest_start
      67                51     2025-01-27   2026-06-02

51 of 67 clients only started search tracking after September 2025.
Same calendar month for everyone, very different amounts of history behind it.


51 of 67. So this is not an edge case, it is most of the panel. Worth remembering before I read
anything into a comparison between two clients.

## Self-check

Before you submit, confirm each line honestly:

* [x] Every section above is filled, thinking written out and code that backs it
* [x] The notebook runs top to bottom with no errors (Runtime, then Run all)
* [x] No client names, web addresses, or private search terms anywhere, and no token in any cell
* [x] My claims stay careful. Observed, measured, provisional, decision support
* [x] Committed under `work/notebooks/`, then submit the repo URL on the card